# Notebook 08 — Crop Data Cleaning (Punjab + Haryana)

| | Punjab | Haryana |
|---|---|---|
| **Source file** | `Punjab_CropProduction_Clean.xlsx` | `Haryana_CropProduction_Clean.xlsx` |
| **Header row** | Row 4 (Excel) → `header=3` | Row 4 (Excel) → `header=3` |
| **State / Year columns** | Present in file | Absent → added manually |
| **Production units** | Tonnes | **000 Tonnes** → × 1 000 |
| **Years covered** | 2010–11 to 2022–23 (multi-year) | 2022–23 only |

**Output:** `Data/Processed/crop_production_clean.csv`  
Schema: `state | district | year | crop | production_t`

In [3]:
import pandas as pd

# Update these paths if files are moved 
PUNJAB_PATH  = r"C:\Users\DSSON\Documents\M.Sc. Sem 2\Residue_Project\Data\Raw\Crop\Punjab Statistical Abstract (Crop Area + Production)\Punjab_CropProduction_Clean.xlsx"
HARYANA_PATH = r"C:\Users\DSSON\Documents\M.Sc. Sem 2\Residue_Project\Data\Raw\Crop\Haryana Statistical Abstract (Crop Area + Production)\Haryana_CropProduction_Clean.xlsx"
OUTPUT_PATH  = r"..\Data\Processed\crop_production_clean.csv"

In [4]:
# Both clean files have 3 header/title rows above the column names,
# so the real header sits on Excel row 4  →  header=3 (0-indexed).
#
# Punjab has two columns called "Area (Ha)", "Production (T)", and
# "Yield (T/Ha)" — one set for Rice, one for Wheat.  pandas will
# auto-suffix duplicates (.1), so we rename by position instead.

punjab_raw = pd.read_excel(PUNJAB_PATH, engine='openpyxl', header=3)

punjab_raw.columns = [
    'state', 'district', 'year',
    'rice_area_ha',  'rice_prod_t',  'rice_yield_t_ha',
    'wheat_area_ha', 'wheat_prod_t', 'wheat_yield_t_ha',
]

print(f"Loaded {len(punjab_raw)} rows")
punjab_raw.head(3)

Loaded 285 rows


,state,district,year,rice_area_ha,rice_prod_t,rice_yield_t_ha,wheat_area_ha,wheat_prod_t,wheat_yield_t_ha
0,Punjab,Amritsar,2010 - 2011,186000.0,503000.0,2.70,189000.0,810000.0,4.29
1,Punjab,Amritsar,2011 - 2012,184000.0,482000.0,2.62,190000.0,945000.0,4.97
2,Punjab,Amritsar,2012 - 2013,185000.0,544000.0,2.94,190000.0,884000.0,4.65


In [5]:
# Haryana's file has 10 columns (Rice + Wheat + Combined + Residue).
# Units are 000 Tonnes / 000 ha — NOT raw Tonnes.
# State and Year are absent from the file and added manually below.

haryana_raw = pd.read_excel(HARYANA_PATH, engine='openpyxl', header=3)

haryana_raw.columns = [
    'district',
    'rice_area_000ha', 'rice_prod_000t',  'rice_yield_kgha',
    'wheat_area_000ha','wheat_prod_000t', 'wheat_yield_kgha',
    'total_area',      'total_prod',      'residue',
]

print(f"Loaded {len(haryana_raw)} rows")
haryana_raw.head(3)

Loaded 23 rows


,district,rice_area_000ha,rice_prod_000t,rice_yield_kgha,wheat_area_000ha,wheat_prod_000t,wheat_yield_kgha,total_area,total_prod,residue
0,Ambala,105.0,390.0,3718.0,89.0,412.0,4638.0,NaN,NaN,NaN
1,Bhiwani,26.3,73.0,2781.0,91.0,371.0,4097.0,NaN,NaN,NaN
2,Charkhi Dadri,7.7,19.0,2445.0,38.0,138.0,3659.0,NaN,NaN,NaN


In [6]:
punjab = punjab_raw.copy()

# Drop the TOTAL row appended at the bottom (no valid district entry)
punjab = punjab[punjab['district'].notna() &
                (punjab['district'].astype(str).str.strip() != '')].copy()

# Ensure numeric production columns
punjab['rice_prod_t']  = pd.to_numeric(punjab['rice_prod_t'],  errors='coerce')
punjab['wheat_prod_t'] = pd.to_numeric(punjab['wheat_prod_t'], errors='coerce')

print(f"Punjab: {punjab.shape[0]} rows, {punjab['district'].nunique()} districts, "
      f"{punjab['year'].nunique()} years")
print(f"Years covered: {sorted(punjab['year'].unique())}")
punjab[['state','district','year','rice_prod_t','wheat_prod_t']].head(4)

Punjab: 284 rows, 23 districts, 13 years
Years covered: ['2010 - 2011', '2011 - 2012', '2012 - 2013', '2013 - 2014', '2014 - 2015', '2015 - 2016', '2016 - 2017', '2017 - 2018', '2018 - 2019', '2019 - 2020', '2020 - 2021', '2021 - 2022', '2022 - 2023']


,state,district,year,rice_prod_t,wheat_prod_t
0,Punjab,Amritsar,2010 - 2011,503000.0,810000.0
1,Punjab,Amritsar,2011 - 2012,482000.0,945000.0
2,Punjab,Amritsar,2012 - 2013,544000.0,884000.0
3,Punjab,Amritsar,2013 - 2014,518000.0,934000.0


In [7]:
haryana = haryana_raw.copy()

# Drop aggregate total row and any blank rows
haryana = haryana[
    haryana['district'].notna() &
    (haryana['district'].astype(str).str.strip() != 'HARYANA TOTAL')
].copy()

# Add state and year (absent from file — single-year snapshot)
haryana['state'] = 'Haryana'
haryana['year']  = '2022-23'

# Unit conversion: Haryana source is in 000 Tonnes → convert to Tonnes
#     to match Punjab's unit before combining.
haryana['rice_prod_t']  = pd.to_numeric(haryana['rice_prod_000t'],  errors='coerce') * 1_000
haryana['wheat_prod_t'] = pd.to_numeric(haryana['wheat_prod_000t'], errors='coerce') * 1_000

print(f"Haryana: {haryana.shape[0]} districts, year = 2022-23")
haryana[['state','district','year','rice_prod_t','wheat_prod_t']].head(4)

Haryana: 22 districts, year = 2022-23


,state,district,year,rice_prod_t,wheat_prod_t
0,Haryana,Ambala,2022-23,390000.0,412000.0
1,Haryana,Bhiwani,2022-23,73000.0,371000.0
2,Haryana,Charkhi Dadri,2022-23,19000.0,138000.0
3,Haryana,Faridabad,2022-23,43000.0,118000.0


In [8]:
ID_COLS  = ['state', 'district', 'year']
PROD_COLS = ['rice_prod_t', 'wheat_prod_t']
CROP_MAP  = {'rice_prod_t': 'Rice', 'wheat_prod_t': 'Wheat'}

def to_long(df: pd.DataFrame) -> pd.DataFrame:
    """Melt rice_prod_t / wheat_prod_t into a single 'crop' + 'production_t' pair."""
    long = (
        df[ID_COLS + PROD_COLS]
        .melt(id_vars=ID_COLS, var_name='crop', value_name='production_t')
    )
    long['crop'] = long['crop'].map(CROP_MAP)
    return (
        long
        .dropna(subset=['production_t'])
        .query('production_t > 0')
        .reset_index(drop=True)
    )

crop_data = pd.concat([to_long(punjab), to_long(haryana)], ignore_index=True)

print(f"Combined shape : {crop_data.shape}")
print(crop_data['state'].value_counts().to_string())
print()
crop_data.head(6)

Combined shape : (611, 5)
state
Punjab     568
Haryana     43



,state,district,year,crop,production_t
0,Punjab,Amritsar,2010 - 2011,Rice,503000.0
1,Punjab,Amritsar,2011 - 2012,Rice,482000.0
2,Punjab,Amritsar,2012 - 2013,Rice,544000.0
3,Punjab,Amritsar,2013 - 2014,Rice,518000.0
4,Punjab,Amritsar,2014 - 2015,Rice,505000.0
5,Punjab,Amritsar,2015 - 2016,Rice,513000.0


In [9]:
assert crop_data['production_t'].isna().sum() == 0, "Unexpected NaNs in production_t"
assert (crop_data['production_t'] > 0).all(),       "Non-positive production values found"
assert set(crop_data['crop'].unique()) == {'Rice','Wheat'}, "Unexpected crop names"

print("All checks passed")
print()
print(crop_data.groupby(['state','crop'])['production_t']
      .agg(['count','sum','mean'])
      .rename(columns={'count':'n_rows','sum':'total_t','mean':'avg_t'})
      .round(0))

✅ All checks passed

               n_rows      total_t     avg_t
state   crop                                
Haryana Rice       21    5920000.0  281905.0
        Wheat      22   11063000.0  502864.0
Punjab  Rice      284  159868000.0  562915.0
        Wheat     284  219785400.0  773892.0


In [10]:
crop_data.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(crop_data)} rows  →  {OUTPUT_PATH}")
print(f"   Columns : {crop_data.columns.tolist()}")

✅ Saved 611 rows  →  ..\Data\Processed\crop_production_clean.csv
   Columns : ['state', 'district', 'year', 'crop', 'production_t']
